# 07 — Bronze DLT: whole-load files (cars, telegram, node_locations, streets_list)

## Configuration


In [0]:
import dlt
from pyspark.sql.functions import current_timestamp, col, lit
from pyspark.sql.types import StructType, StructField, StringType

CATALOG = spark.conf.get("catalog_name", "vstone_catalog")
RAW_SCHEMA = spark.conf.get("raw_schema", "raw")
LANDING_VOL = spark.conf.get("landing_volume", "landing")
LANDING_PATH = f"/Volumes/{CATALOG}/{RAW_SCHEMA}/{LANDING_VOL}"

CARS_SCHEMA = StructType([
    StructField("enter", StringType(), True), StructField("exit", StringType(), True),
    StructField("date", StringType(), True), StructField("id", StringType(), True),
    StructField("location", StringType(), True),
])

BRONZE_PROPS = {
    "quality": "bronze",
    "delta.enableChangeDataFeed": "true",
    "pipelines.reset.allowed": "true",
}

## cars_dlt

In [0]:
CARS_FILE = "cars.csv"

@dlt.table(
    name="cars_dlt",
    comment="Bronze: cars.csv (whole load — not chunked, see data_mapping.md) via DLT.",
    table_properties=BRONZE_PROPS,
)
@dlt.expect("valid_location", "location IS NOT NULL")
@dlt.expect("valid_date", "date IS NOT NULL")
def cars_dlt():
    return (spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.inferColumnTypes", "false")
            .option("cloudFiles.schemaEvolutionMode", "none")
            .option("header", "true")
            .option("pathGlobFilter", CARS_FILE)
            .schema(CARS_SCHEMA)
            .load(LANDING_PATH)
            .withColumn("load_dt", current_timestamp())
            .withColumn("source", col("_metadata.file_name")))

## telegram_dlt

In [0]:
TELEGRAM_FILE = "telegram.csv"

@dlt.table(
    name="telegram_dlt",
    comment="Bronze: telegram.csv (whole load, multi-line quoted free text) via DLT.",
    table_properties=BRONZE_PROPS,
)
@dlt.expect("valid_message", "message IS NOT NULL")
def telegram_dlt():
    return (spark.readStream.format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.inferColumnTypes", "false")
            .option("cloudFiles.schemaEvolutionMode", "none")
            .option("header", "true")
            .option("multiLine", "true")
            .option("escape", '"')
            .option("quote", '"')
            .option("pathGlobFilter", TELEGRAM_FILE)
            .load(LANDING_PATH)
            .withColumn("load_dt", current_timestamp())
            .withColumn("source", col("_metadata.file_name")))

## node_locations_dlt

In [0]:
@dlt.table(
    name="node_locations_dlt",
    comment="Bronze: node_locations.csv (14-row dimension) via DLT. location=7 has known (0,0) coords — see assumptions doc.",
    table_properties=BRONZE_PROPS,
)
@dlt.expect("valid_location", "location IS NOT NULL")
def node_locations_dlt():
    return (spark.read.format("csv").option("header", "true")
            .load(f"{LANDING_PATH}/node_locations.csv")
            .withColumn("load_dt", current_timestamp())
            .withColumn("source", lit("node_locations.csv")))

##streets_list_dlt

In [0]:
@dlt.table(
    name="streets_list_dlt",
    comment="Bronze: streets_list.csv (36-row dimension) via DLT. street_id is the FK target for streets_dlt/streets_autoloader/streets_xml.",
    table_properties=BRONZE_PROPS,
)
@dlt.expect("valid_street_id", "street_id IS NOT NULL")
def streets_list_dlt():
    return (spark.read.format("csv").option("header", "true")
            .load(f"{LANDING_PATH}/streets_list.csv")
            .withColumn("load_dt", current_timestamp())
            .withColumn("source", lit("streets_list.csv")))